In [13]:
import os
import json
import re
import string
import transformers

import nltk
import pandas as pd
from nltk.corpus import stopwords
from transformers import BertTokenizer

nltk.download('stopwords')

tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p2")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\elang\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
# Load the AES dataset from CSV file
file_path = "dataset/new_dataset.csv"
df = pd.read_csv(file_path)

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nSample of the dataset:")
print(df.head(10))

    # Check unique Mata Pelajaran
print(f"\nUnique Mata Pelajaran: {df['Mata Pelajaran'].unique()}")
print("Jumlah per Mata Pelajaran:")
print(df['Mata Pelajaran'].value_counts())

Dataset shape: (1460, 5)
Columns: ['Mata Pelajaran', 'Soal', 'Jawaban', 'Skor', 'Keterangan']

Sample of the dataset:
  Mata Pelajaran                                               Soal  \
0           IPAS                            Jelaskan Pengertian Zat   
1           IPAS  Apa perbedaan antara perubahan fisika dan peru...   
2           IPAS  Mengapa pakan ternak yang membusuk termasuk pe...   
3           IPAS      Sebutkan ciri-ciri terjadinya perubahan kimia   
4           IPAS  Gula yang dilarutkan dalam air tidak terlihat ...   
5           IPAS  Air yang membeku menjadi es sering dikatakan b...   
6           IPAS  Mengapa pembakaran kertas menghasilkan zat bar...   
7           IPAS  Jika suatu zat mengalami perubahan warna, apak...   
8           IPAS                            Jelaskan Pengertian Zat   
9           IPAS  Apa perbedaan antara perubahan fisika dan peru...   

                                             Jawaban  Skor     Keterangan  
0  Zat yaitu segala sesu

In [15]:
slangwords = {'pd' : 'pada','yg' : 'yang','kpd' : 'kepada','tsb' : 'tersebut','tdk' : 'tidak','dn' : 'dan','dgn' : 'dengan','hub' : 'hubungan','dg' : 'dengan','dlm' : 'dalam','dpt' : 'dapat','org' : 'orang','sgt' : 'sangat','dll' : 'dan lain lain'}

In [16]:
def cleaningText(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove mentions, hashtags, URLs
    text = re.sub(r'@[A-Za-z0-9]+', ' ', text)
    text = re.sub(r'#[A-Za-z0-9]+', ' ', text)
    text = re.sub(r"http\S+", '', text)
    # Remove numbers and special characters, keep letters and spaces
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

def casefoldingText(text):
    """Convert text to lowercase"""
    return (text or '').lower()

def fix_slangwords(text):
    """Replace slangwords with formal equivalents"""
    if not isinstance(text, str):
        return ""
        
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    return ' '.join(fixed_words)

def process_text_column(text):
    """Apply all text processing steps"""
    cleaned = cleaningText(text)
    casefolded = casefoldingText(cleaned)
    slang_fixed = fix_slangwords(casefolded)
    return slang_fixed

In [17]:
# Apply text processing to relevant columns
print("Processing text columns...")

# Process Mata Pelajaran
df['Mata Pelajaran_clean'] = df['Mata Pelajaran'].apply(process_text_column)

# Process Soal
df['Soal_clean'] = df['Soal'].apply(process_text_column)

# Process Jawaban
df['Jawaban_clean'] = df['Jawaban'].apply(process_text_column)

# Process Keterangan
df['Keterangan_clean'] = df['Keterangan'].apply(process_text_column)

print("Text processing completed!")
print("\nSample of processed data:")
print(df[['Mata Pelajaran_clean', 'Soal_clean', 'Jawaban_clean', 'Keterangan_clean']].head())

Processing text columns...
Text processing completed!

Sample of processed data:
  Mata Pelajaran_clean                                         Soal_clean  \
0                 ipas                            jelaskan pengertian zat   
1                 ipas  apa perbedaan antara perubahan fisika dan peru...   
2                 ipas  mengapa pakan ternak yang membusuk termasuk pe...   
3                 ipas      sebutkan ciri ciri terjadinya perubahan kimia   
4                 ipas  gula yang dilarutkan dalam air tidak terlihat ...   

                                       Jawaban_clean Keterangan_clean  
0  zat yaitu segala sesuatu yang menempati ruang ...    jawaban siswa  
1  perubahan fisika perubahan yang tidak menghasi...    jawaban siswa  
2  termasuk zat kimia karena terbentuknya zat bar...    jawaban siswa  
3           membentuk zat baru berubah bentuk ukuran    jawaban siswa  
4  perubahan fisika karena gula dalam air akan me...    jawaban siswa  


In [18]:
from sklearn.model_selection import train_test_split

def remove_duplicate_question_answers(df):
    """Remove rows where both question and answer are identical"""
    original_size = len(df)
    
    # Create a combined key of question + answer to identify exact duplicates
    df['qa_key'] = df['Soal_clean'] + '|||' + df['Jawaban_clean']
    
    # Remove duplicates, keeping first occurrence
    df_unique = df.drop_duplicates(subset=['qa_key'], keep='first')
    
    # Remove the temporary key column
    df_unique = df_unique.drop('qa_key', axis=1)
    
    removed_count = original_size - len(df_unique)
    print(f"Removed {removed_count} duplicate question-answer pairs")
    print(f"Original size: {original_size}, After deduplication: {len(df_unique)}")
    
    return df_unique

def split_data_balanced_70_15_15(df, random_state=42):
    """
    Split data into 70% train, 15% validation, 15% test with balanced distribution
    across Mata Pelajaran and shared questions between sets
    """
    # First remove duplicate question-answer pairs
    df_clean = remove_duplicate_question_answers(df)
    
    # Group by question to identify unique questions
    question_groups = df_clean.groupby('Soal_clean')
    unique_questions = list(question_groups.groups.keys())
    
    print(f"Total unique questions: {len(unique_questions)}")
    
    # Create question pools for different overlap scenarios
    np.random.seed(random_state)
    
    # 40% of questions will be shared across all three sets
    # 30% shared between train+validation only  
    # 30% train only
    n_all_shared = int(len(unique_questions) * 0.4)
    n_train_val_shared = int(len(unique_questions) * 0.3)
    
    all_shared_questions = np.random.choice(unique_questions, n_all_shared, replace=False)
    remaining_questions = [q for q in unique_questions if q not in all_shared_questions]
    train_val_shared_questions = np.random.choice(remaining_questions, n_train_val_shared, replace=False)
    train_only_questions = [q for q in remaining_questions if q not in train_val_shared_questions]
    
    print(f"Questions shared across all sets: {len(all_shared_questions)}")
    print(f"Questions shared train+val only: {len(train_val_shared_questions)}")
    print(f"Questions train only: {len(train_only_questions)}")
    
    # Collect data for each set
    train_data = []
    val_data = []
    test_data = []
    
    # Process each question group
    for question, group in question_groups:
        # Balance by Mata Pelajaran within each question
        mp_groups = group.groupby('Mata Pelajaran_clean')
        
        for mp, mp_group in mp_groups:
            mp_data = mp_group.copy()
            
            if question in all_shared_questions:
                # This question appears in all three sets
                n_samples = len(mp_group)
                if n_samples >= 3:
                    # Distribute as evenly as possible
                    test_size = max(1, n_samples // 3)
                    val_size = max(1, (n_samples - test_size) // 2)
                    train_size = n_samples - test_size - val_size
                    
                    # Randomly assign samples
                    indices = mp_group.index.tolist()
                    np.random.shuffle(indices)
                    
                    test_indices = indices[:test_size]
                    val_indices = indices[test_size:test_size + val_size]
                    train_indices = indices[test_size + val_size:]
                    
                    test_data.append(mp_group.loc[test_indices])
                    val_data.append(mp_group.loc[val_indices])
                    train_data.append(mp_group.loc[train_indices])
                elif n_samples == 2:
                    # Split between train and test, validation gets from other questions
                    test_sample = mp_group.sample(1, random_state=random_state)
                    train_sample = mp_group.drop(test_sample.index)
                    test_data.append(test_sample)
                    train_data.append(train_sample)
                else:
                    # Only one sample, put in train
                    train_data.append(mp_group)
                    
            elif question in train_val_shared_questions:
                # This question appears in train and validation only
                if len(mp_group) >= 2:
                    val_sample = mp_group.sample(1, random_state=random_state)
                    train_sample = mp_group.drop(val_sample.index)
                    val_data.append(val_sample)
                    train_data.append(train_sample)
                else:
                    train_data.append(mp_group)
            else:
                # This question appears only in train
                train_data.append(mp_group)
    
    # Combine all data
    train_df = pd.concat(train_data, ignore_index=True)
    val_df = pd.concat(val_data, ignore_index=True)
    test_df = pd.concat(test_data, ignore_index=True)
    
    # Balance each dataset to achieve 70:15:15 ratio and equal MP distribution
    total_size = len(df_clean)
    target_train_size = int(total_size * 0.7)
    target_val_size = int(total_size * 0.15)
    target_test_size = total_size - target_train_size - target_val_size
    
    def balance_to_target_size(dataset, target_size):
        """Balance dataset to target size with equal MP distribution"""
        current_size = len(dataset)
        if current_size <= target_size:
            return dataset
        
        mp_groups = dataset.groupby('Mata Pelajaran_clean')
        target_per_mp = target_size // 2  # Equal distribution between 2 MPs
        
        balanced_parts = []
        for mp, mp_group in mp_groups:
            if len(mp_group) > target_per_mp:
                mp_group = mp_group.sample(target_per_mp, random_state=random_state)
            balanced_parts.append(mp_group)
        
        result = pd.concat(balanced_parts, ignore_index=True)
        
        # If still too large, sample randomly
        if len(result) > target_size:
            result = result.sample(target_size, random_state=random_state)
        
        return result
    
    # Balance each dataset
    train_df = balance_to_target_size(train_df, target_train_size)
    val_df = balance_to_target_size(val_df, target_val_size)
    test_df = balance_to_target_size(test_df, target_test_size)
    
    return train_df, val_df, test_df

# Import numpy for random operations
import numpy as np

# Split the data with 70:15:15 ratio
train_df, val_df, test_df = split_data_balanced_70_15_15(df)

print(f"\n=== FINAL DATASET SIZES ===")
print(f"Train set size: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation set size: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set size: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

print(f"\n=== MATA PELAJARAN DISTRIBUTION ===")
print("Train set:")
print(train_df['Mata Pelajaran_clean'].value_counts())
print("\nValidation set:")
print(val_df['Mata Pelajaran_clean'].value_counts())
print("\nTest set:")
print(test_df['Mata Pelajaran_clean'].value_counts())

Removed 103 duplicate question-answer pairs
Original size: 1460, After deduplication: 1357
Total unique questions: 48
Questions shared across all sets: 19
Questions shared train+val only: 14
Questions train only: 15

=== FINAL DATASET SIZES ===
Train set size: 949 (65.0%)
Validation set size: 194 (13.3%)
Test set size: 174 (11.9%)

=== MATA PELAJARAN DISTRIBUTION ===
Train set:
Mata Pelajaran_clean
pendidikan pancasila      236
ipas                      205
seni                      196
bahasa indonesia          171
pendidikan agama islam    141
Name: count, dtype: int64

Validation set:
Mata Pelajaran_clean
pendidikan agama islam    64
bahasa indonesia          48
seni                      44
ipas                      26
pendidikan pancasila      12
Name: count, dtype: int64

Test set:
Mata Pelajaran_clean
pendidikan agama islam    63
bahasa indonesia          45
seni                      38
ipas                      20
pendidikan pancasila       8
Name: count, dtype: int64


In [19]:
# Sort each dataset by Mata Pelajaran and Soal
def sort_datasets(train_df, val_df, test_df):
    """
    Sort datasets by Mata Pelajaran and Soal to ensure consistent ordering
    """
    # Create a copy to avoid SettingWithCopyWarning
    train_sorted = train_df.copy()
    val_sorted = val_df.copy()
    test_sorted = test_df.copy()
    
    # Sort by Mata Pelajaran_clean and Soal_clean
    train_sorted = train_sorted.sort_values(['Mata Pelajaran_clean', 'Soal_clean'])
    val_sorted = val_sorted.sort_values(['Mata Pelajaran_clean', 'Soal_clean'])
    test_sorted = test_sorted.sort_values(['Mata Pelajaran_clean', 'Soal_clean'])
    
    return train_sorted, val_sorted, test_sorted

# Sort the datasets
train_sorted, val_sorted, test_sorted = sort_datasets(train_df, val_df, test_df)

print("Datasets sorted by Mata Pelajaran and Soal")
print("\nSample of sorted train data:")
print(train_sorted[['Mata Pelajaran_clean', 'Soal_clean', 'Jawaban_clean']].head(10))

Datasets sorted by Mata Pelajaran and Soal

Sample of sorted train data:
   Mata Pelajaran_clean                                         Soal_clean  \
23     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
10     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
25     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
2      bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
5      bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
9      bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
7      bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
0      bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
6      bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
24     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   

                                        Jawaban_clean  
23          

In [20]:
# Create output directory
output_dir = "data_clean_aes"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Define columns to save
output_columns = [
    'Mata Pelajaran_clean', 
    'Soal_clean', 
    'Jawaban_clean', 
    'Keterangan_clean',
    'Skor'  # Keep original score for reference
]

# Save each dataset
train_path = os.path.join(output_dir, "train_data.csv")
val_path = os.path.join(output_dir, "val_data.csv")
test_path = os.path.join(output_dir, "test_data.csv")

train_sorted[output_columns].to_csv(train_path, index=False, encoding='utf-8-sig')
val_sorted[output_columns].to_csv(val_path, index=False, encoding='utf-8-sig')
test_sorted[output_columns].to_csv(test_path, index=False, encoding='utf-8-sig')

print(f"Train data saved to: {train_path}")
print(f"Validation data saved to: {val_path}")
print(f"Test data saved to: {test_path}")

# Save summary statistics
summary_stats = {
    'total_samples': len(df),
    'train_samples': len(train_sorted),
    'val_samples': len(val_sorted),
    'test_samples': len(test_sorted),
    'mata_pelajaran_distribution': {
        'train': train_sorted['Mata Pelajaran_clean'].value_counts().to_dict(),
        'val': val_sorted['Mata Pelajaran_clean'].value_counts().to_dict(),
        'test': test_sorted['Mata Pelajaran_clean'].value_counts().to_dict()
    }
}

summary_path = os.path.join(output_dir, "data_summary.json")
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_stats, f, indent=2, ensure_ascii=False)

print(f"\nData summary saved to: {summary_path}")
print("\nProcessing completed successfully!")

Train data saved to: data_clean_aes\train_data.csv
Validation data saved to: data_clean_aes\val_data.csv
Test data saved to: data_clean_aes\test_data.csv

Data summary saved to: data_clean_aes\data_summary.json

Processing completed successfully!


In [21]:
# Display final processed data samples
print("=== FINAL PROCESSED DATA SAMPLES ===\n")

print("Train set sample:")
print(train_sorted[['Mata Pelajaran_clean', 'Soal_clean', 'Jawaban_clean']].head(3))
print("\n" + "="*50 + "\n")

print("Validation set sample:")
print(val_sorted[['Mata Pelajaran_clean', 'Soal_clean', 'Jawaban_clean']].head(3))
print("\n" + "="*50 + "\n")

print("Test set sample:")
print(test_sorted[['Mata Pelajaran_clean', 'Soal_clean', 'Jawaban_clean']].head(3))
print("\n" + "="*50 + "\n")

# Display summary
print("SUMMARY:")
print(f"Total data processed: {len(df)}")
print(f"Train: {len(train_sorted)} samples")
print(f"Validation: {len(val_sorted)} samples") 
print(f"Test: {len(test_sorted)} samples")
print(f"Output directory: {output_dir}")

=== FINAL PROCESSED DATA SAMPLES ===

Train set sample:
   Mata Pelajaran_clean                                         Soal_clean  \
23     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
10     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   
25     bahasa indonesia  apa tujuan dibuatnya teks laporan hasil observ...   

                                        Jawaban_clean  
23                   agar dapat mengenali benda asing  
10  menyapaikan hasil pengamatan scara jelas dan f...  
25  memberikan informasi yang objektif faktual dan...  


Validation set sample:
   Mata Pelajaran_clean                                         Soal_clean  \
10     bahasa indonesia  apa yang dimaksud dengan hikayat jelaskan deng...   
11     bahasa indonesia  apa yang dimaksud dengan hikayat jelaskan deng...   
12     bahasa indonesia  apa yang dimaksud dengan hikayat jelaskan deng...   

                                        Jawaban_clean  
10  hikayat a

In [22]:
# Enhanced verification - Check question overlap and distribution
def check_question_overlap_and_distribution(train_df, val_df, test_df):
    """Check question overlap and ensure proper distribution"""
    train_questions = set(train_df['Soal_clean'])
    val_questions = set(val_df['Soal_clean'])
    test_questions = set(test_df['Soal_clean'])
    
    train_val_overlap = train_questions & val_questions
    train_test_overlap = train_questions & test_questions
    val_test_overlap = val_questions & test_questions
    all_three_overlap = train_questions & val_questions & test_questions
    
    print("=== QUESTION OVERLAP ANALYSIS ===")
    print(f"Total unique questions in train: {len(train_questions)}")
    print(f"Total unique questions in validation: {len(val_questions)}")
    print(f"Total unique questions in test: {len(test_questions)}")
    
    print(f"\nTrain-Val overlap: {len(train_val_overlap)} questions")
    print(f"Train-Test overlap: {len(train_test_overlap)} questions")
    print(f"Val-Test overlap: {len(val_test_overlap)} questions")
    print(f"All three sets overlap: {len(all_three_overlap)} questions")
    
    print(f"\nQuestions only in train: {len(train_questions - val_questions - test_questions)}")
    print(f"Questions only in validation: {len(val_questions - train_questions - test_questions)}")
    print(f"Questions only in test: {len(test_questions - train_questions - val_questions)}")
    
    # Show sample shared questions
    if all_three_overlap:
        print(f"\nSample questions shared across all sets:")
        for i, q in enumerate(list(all_three_overlap)[:3]):
            print(f"  {i+1}. {q[:100]}...")
    
    # Check distribution by Mata Pelajaran for shared questions
    if all_three_overlap:
        print(f"\n=== DISTRIBUTION OF SHARED QUESTIONS BY MATA PELAJARAN ===")
        shared_df = df[df['Soal_clean'].isin(all_three_overlap)]
        shared_dist = shared_df.groupby(['Soal_clean', 'Mata Pelajaran_clean']).size().unstack(fill_value=0)
        print("Sample of shared questions distribution:")
        print(shared_dist.head(10))

check_question_overlap_and_distribution(train_sorted, val_sorted, test_sorted)

=== QUESTION OVERLAP ANALYSIS ===
Total unique questions in train: 48
Total unique questions in validation: 33
Total unique questions in test: 19

Train-Val overlap: 33 questions
Train-Test overlap: 19 questions
Val-Test overlap: 19 questions
All three sets overlap: 19 questions

Questions only in train: 15
Questions only in validation: 0
Questions only in test: 0

Sample questions shared across all sets:
  1. sebutkan tiga contoh akhlaq mazmumah di kalangan remaja sekarang dan berikan contoh perilaku yang me...
  2. pilih salah satu fastabiqul khoirot syu abul iman atau akhlaq mazmumah jelaskan bagaimana penerapan ...
  3. mengapa karya seni rupa dua dimensi memiliki peluang besar dalam industri kreatif...

=== DISTRIBUTION OF SHARED QUESTIONS BY MATA PELAJARAN ===
Sample of shared questions distribution:
Mata Pelajaran_clean                                bahasa indonesia  ipas  \
Soal_clean                                                                   
apa yang dimaksud dengan a

In [23]:
# Enhanced verification - Check 70:15:15 split and question overlap
def verify_split_ratio_and_overlap(train_df, val_df, test_df, original_df):
    """Verify the 70:15:15 split ratio and question overlap"""
    
    total_size = len(original_df)
    train_pct = len(train_df) / total_size * 100
    val_pct = len(val_df) / total_size * 100
    test_pct = len(test_df) / total_size * 100
    
    print("=== SPLIT RATIO VERIFICATION ===")
    print(f"Target ratio: 70% : 15% : 15%")
    print(f"Actual ratio: {train_pct:.1f}% : {val_pct:.1f}% : {test_pct:.1f}%")
    
    # Check if ratios are close to target
    tolerance = 2.0  # 2% tolerance
    if abs(train_pct - 70) <= tolerance and abs(val_pct - 15) <= tolerance and abs(test_pct - 15) <= tolerance:
        print("✅ Split ratio is within acceptable range")
    else:
        print("⚠️ Split ratio may need adjustment")
    
    print(f"\n=== QUESTION OVERLAP ANALYSIS ===")
    train_questions = set(train_df['Soal_clean'])
    val_questions = set(val_df['Soal_clean'])
    test_questions = set(test_df['Soal_clean'])
    
    train_val_overlap = train_questions & val_questions
    train_test_overlap = train_questions & test_questions
    val_test_overlap = val_questions & test_questions
    all_three_overlap = train_questions & val_questions & test_questions
    
    print(f"Train-Val overlap: {len(train_val_overlap)} questions ({len(train_val_overlap)/len(train_questions)*100:.1f}% of train)")
    print(f"Train-Test overlap: {len(train_test_overlap)} questions ({len(train_test_overlap)/len(train_questions)*100:.1f}% of train)")
    print(f"Val-Test overlap: {len(val_test_overlap)} questions")
    print(f"All three sets overlap: {len(all_three_overlap)} questions")
    
    print(f"\n=== MATA PELAJARAN BALANCE CHECK ===")
    def check_balance(df, set_name):
        mp_counts = df['Mata Pelajaran_clean'].value_counts()
        if len(mp_counts) >= 2:
            ratio = mp_counts.iloc[0] / mp_counts.iloc[1]
            print(f"{set_name} - Ratio: {ratio:.2f}", end=" ")
            if 0.8 <= ratio <= 1.2:
                print("✅ Balanced")
            else:
                print("⚠️ Unbalanced")
        else:
            print(f"{set_name} - Only one subject found")
    
    check_balance(train_df, "Train")
    check_balance(val_df, "Validation")
    check_balance(test_df, "Test")
    
    print(f"\n=== DUPLICATE QUESTION-ANSWER CHECK ===")
    # Check for any remaining duplicates within each set
    for name, df_set in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        df_set['qa_key'] = df_set['Soal_clean'] + '|||' + df_set['Jawaban_clean']
        duplicates = df_set[df_set.duplicated(subset=['qa_key'], keep=False)]
        if len(duplicates) > 0:
            print(f"⚠️ {name} set has {len(duplicates)} duplicate question-answer pairs")
        else:
            print(f"✅ {name} set has no duplicate question-answer pairs")
        df_set = df_set.drop('qa_key', axis=1)

# Run verification
verify_split_ratio_and_overlap(train_df, val_df, test_df, df)

=== SPLIT RATIO VERIFICATION ===
Target ratio: 70% : 15% : 15%
Actual ratio: 65.0% : 13.3% : 11.9%
⚠️ Split ratio may need adjustment

=== QUESTION OVERLAP ANALYSIS ===
Train-Val overlap: 33 questions (68.8% of train)
Train-Test overlap: 19 questions (39.6% of train)
Val-Test overlap: 19 questions
All three sets overlap: 19 questions

=== MATA PELAJARAN BALANCE CHECK ===
Train - Ratio: 1.15 ✅ Balanced
Validation - Ratio: 1.33 ⚠️ Unbalanced
Test - Ratio: 1.40 ⚠️ Unbalanced

=== DUPLICATE QUESTION-ANSWER CHECK ===
✅ Train set has no duplicate question-answer pairs
✅ Validation set has no duplicate question-answer pairs
✅ Test set has no duplicate question-answer pairs
